## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

In [1]:
# The imports
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


True

In [3]:
# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")


In [ ]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-4o-mini")

In [4]:
# Run the joke with Runner.run(agent, prompt) then print final_output

result = await Runner.run(agent, "Tell me a joke about Autonomous AI agents")

# with trace("Telling a joke"):
#     result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
#     print(result.final_output)

In [5]:
# return final output

print(result.final_output)

Why did the autonomous AI agent break up with their partner?

Because they couldn't find common ground... they kept running in circles!


In [6]:
# Print the details of the LLM call

result.to_input_list()


[{'content': 'Tell me a joke about Autonomous AI agents', 'role': 'user'},
 {'id': 'msg_0802d83731b89116006a404a230870819bba6de9dc4efebab9',
  'content': [{'annotations': [],
    'text': "Why did the autonomous AI agent break up with their partner?\n\nBecause they couldn't find common ground... they kept running in circles!",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

## Adding Observability With a Trace


In [7]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
    print(result.final_output)

# Print the details of the LLM call

# result.to_input_list()

Why did the autonomous AI agent break up with its partner?

Because it couldn’t find the right "algorithm" for love!


## Now go and look at the trace

https://platform.openai.com/traces

In [8]:
# Streaming 

result = Runner.run_streamed(agent, input="Tell me a joke about Autonomous AI Agents.")

async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

# Print the final output


Why did the autonomous AI agent break up with its partner?

Because it found someone with better "algorithm compatibility"!

## Part 2: Adding a tool

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token =os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushiover user is great!")
    else:
        print("Pushover user is not valid. Please start with 'u='")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token is great!")
    else:
        print("Pushover token is not valid. Please start with 'o='");
else: print("Pushover Token not found")



        

Pushiover user is great!
Pushover token is great!


In [12]:
# Push function

def push(message):
    print(f"push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [13]:
push("Hello, world!")

push: Hello, world!


In [14]:
@function_tool
def push_tool(message: str) -> str:
    """Send a pushover notification to the user"""
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with api status code {result }"

In [16]:
# Create Notifier Agent

notifier = Agent(name="Notifier", model="gpt-4o-mini", instructions="Notify the user upon request", tools=[push_tool])

In [17]:
with trace("Pizza has arrrived!"):
    result = await Runner.run(notifier, "Notifiy the user that the pizza is here")
    
    print(result.final_output)

The user has been notified that their pizza has arrived! 🍕


## Part 3: Sessions AKA Memory !!

within a Runnner.run() application level turn, the history/memory is maintained.

But each call to Runner.run() is a fresh start

See example below:

In [18]:
agent = Agent(name="Assistant", model="gpt-4o-mini")

In [19]:
response = await Runner.run(agent, "Hi, there my name is Yohanes")
print(response.final_output)

Hi Yohanes! How can I assist you today?


In [26]:
response = await Runner.run(agent, "Hey, What's my name?")
print(response.final_output)

I can't see your name. If you tell me, I’ll remember it for our chat!


## Memory approach 1 - just manually pass in a list of discs

In [ ]:
# MAnually add structured context from history for the next promp

response.to_input_list()

[{'content': "Hey, What's my name?", 'role': 'user'},
 {'id': 'msg_027021041f3172cb006a40689c7a5c81998a34339af5c69b2b',
  'content': [{'annotations': [],
    'text': "I can't see your name. If you tell me, I’ll remember it for our chat!",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'}]

In [30]:
next_input = response.to_input_list() + [{"role": "user", "content": "what is my name?"}]
next_input

[{'content': "Hey, What's my name?", 'role': 'user'},
 {'id': 'msg_027021041f3172cb006a40689c7a5c81998a34339af5c69b2b',
  'content': [{'annotations': [],
    'text': "I can't see your name. If you tell me, I’ll remember it for our chat!",
    'type': 'output_text',
    'logprobs': []}],
  'role': 'assistant',
  'status': 'completed',
  'type': 'message'},
 {'role': 'user', 'content': 'what is my name?'}]

In [32]:
response = await Runner.run(agent, next_input)
print(response.final_output)

I don't know your name yet! If you share it with me, I’ll remember it for our conversation.


## Another (best) Approach - use OpenAI Agents SDK built in SQLLite session

In [33]:
# this is created in-memory
# For an on-disk memory, use SQLiteSession("1234", "memory.db")
# first parameter is session id (could be username, user email, some unique value to the client) and second is the database to store context
# for this example, we'll just hardcode a userID 
# this doesn't affect the agent, this is strictly for when using Runner.run()

session = SQLiteSession("1234")

In [35]:
response = await Runner.run(agent, "Hi, my name is Yohanes.", session=session)
print(response.final_output)

Hi again, Yohanes! How’s your day going?


In [36]:
response = await Runner.run(agent, "Hello, do you remember my name now?", session=session)
print(response.final_output)

Yes, I remember your name is Yohanes! How can I help you today?


## Conclusion

Agents, Runners (agent loop), traces (observability), streaming, function tools, memory.
Many of the light-eeight agent frameworks are very similar to this construct so this is a great reference point moving forward!
Feel free to checkout the docs at https://openai.github.io/openai-agents-python